## Random Forest Baseline

This notebook trains the first traditional machine learning baseline model using the prepared NASA RUL dataset.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL


In [5]:
import pandas as pd

from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer

from src.preprocessing.feature_selector import FeatureSelector
from src.preprocessing.data_splitter import DataSplitter
from src.preprocessing.feature_scaler import FeatureScaler
from src.preprocessing.preparation_pipeline import DataPreparationPipeline
from src.evaluation.evaluator import RegressionEvaluator
from src.utils.constant import SENSOR_COLUMNS

from src.config.config import (
    TRAIN_DATA_PATH,
    TEST_DATA_PATH,
    RUL_DATA_PATH
)

1- LOAD DATA

In [6]:
loader = DataLoader(
    train_path=TRAIN_DATA_PATH,
    test_path=TEST_DATA_PATH,
    rul_path=RUL_DATA_PATH,
)

train_df = loader.load_train()
test_df = loader.load_test()
rul_df = loader.load_rul()

2026-08-04 00:18:12 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-08-04 00:18:13 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-08-04 00:18:13 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-08-04 00:18:14 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-08-04 00:18:14 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-08-04 00:18:14 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully


2- VALIDATE DATA

In [7]:
validator = DataValidator(
    train_df,
    test_df,
    rul_df
)

validator.validate_all()

2026-08-04 00:18:14 | INFO | validator.py | Line:40 | Validating training dataset...
2026-08-04 00:18:14 | INFO | validator.py | Line:50 | Validating testing dataset...
2026-08-04 00:18:14 | INFO | validator.py | Line:60 | Validating RUL dataset...


{'train': {'valid': True, 'errors': [], 'warnings': []},
 'test': {'valid': True, 'errors': [], 'warnings': []},
 'rul': {'valid': True, 'errors': [], 'warnings': ['Duplicate rows found.']}}

3- Generate RUL

In [8]:
generator = RULGenerator(train_df=train_df)

train_df = generator.generate(cap=125)

train_df.head()

2026-08-04 00:18:14 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-08-04 00:18:14 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 125
2026-08-04 00:18:14 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,125


4- Feature Engineering

In [9]:
engineer = FeatureEngineer(sensor_columns=SENSOR_COLUMNS,lags=[1,2,3],)

feature_df = engineer.transform(train_df)
feature_df.head()

2026-08-04 00:18:14 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-08-04 00:18:14 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-08-04 00:18:15 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-08-04 00:18:17 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
2026-08-04 00:18:17 | INFO | feature_engineer.py | Line:205 | Generating Rate of Change features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:213: PerformanceWarning: DataFrame

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,182.81,-0.26,-28.70,-0.1422,0.0,31.0,112.0,0.00,13.75,8.2882
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,-182.97,0.24,20.49,0.2094,0.0,-32.0,-112.0,0.00,-13.89,-8.2339
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,0.18,0.05,9.43,-0.0638,0.0,-1.0,0.0,0.00,0.06,-0.0037
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,34.31,-359.94,-210.25,1.4997,0.0,-23.0,-297.0,-15.07,3.49,2.2578


5- Create Pipeline

In [10]:
pipeline = DataPreparationPipeline(

    splitter=DataSplitter(
        test_size=0.2,
    ),

    selector=FeatureSelector(

        target_column="RUL",

        drop_columns=[
            "unit_number",
        ],

    ),

    scaler=FeatureScaler(),

)

In [11]:
X_train, X_val, y_train, y_val = pipeline.prepare(
    feature_df
)

2026-08-04 00:18:17 | INFO | preparation_pipeline.py | Line:37 | Starting Data Preparation Pipeline...
2026-08-04 00:18:17 | INFO | data_splitter.py | Line:35 | Starting engine-based train/validation split...
2026-08-04 00:18:17 | INFO | data_splitter.py | Line:67 | Train Engines: 199 | Validation Engines: 50
2026-08-04 00:18:17 | INFO | data_splitter.py | Line:72 | Data splitting completed successfully.
2026-08-04 00:18:17 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-08-04 00:18:17 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-08-04 00:18:17 | INFO | feature_selector.py | Line:29 | Starting Feature Selection...
2026-08-04 00:18:17 | INFO | feature_selector.py | Line:40 | Feature Selection completed successfully.
2026-08-04 00:18:17 | INFO | feature_scaler.py | Line:33 | Fitting Feature Scaler...
2026-08-04 00:18:18 | INFO | feature_scaler.py | Line:37 | Feature Scaler fitted successfully.
2026-08-04 00:18:18 | IN

In [12]:
print("Training Features :", X_train.shape)
print("Validation Features :", X_val.shape,"\n")

print("Training Target :", y_train.shape)
print("Validation Target :", y_val.shape)

Training Features : (49294, 151)
Validation Features : (11955, 151) 

Training Target : (49294,)
Validation Target : (11955,)


In [13]:
from src.models.traditional_ml_benchmark import TraditionalMLBenchmark

benchmark = TraditionalMLBenchmark()

results, best_trainer = benchmark.run(
    X_train,
    y_train,
    X_val,
    y_val,
)

results

2026-08-04 00:18:22 | INFO | traditional_ml_benchmark.py | Line:35 | Starting Traditional ML Benchmark...
2026-08-04 00:18:22 | INFO | traditional_ml_benchmark.py | Line:44 | Training random_forest...
2026-08-04 00:18:22 | INFO | base_trainer.py | Line:24 | Training RandomForestRegressor...
2026-08-04 00:22:40 | INFO | base_trainer.py | Line:30 | Training completed successfully.
2026-08-04 00:22:40 | INFO | base_trainer.py | Line:37 | Generating predictions using RandomForestRegressor...
2026-08-04 00:22:41 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-08-04 00:22:41 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026-08-04 00:22:41 | INFO | traditional_ml_benchmark.py | Line:67 | random_forest | RMSE=19.9101 | R2=0.7628
2026-08-04 00:22:41 | INFO | traditional_ml_benchmark.py | Line:44 | Training xgboost...
2026-08-04 00:22:41 | INFO | base_trainer.py | Line:24 | Training XGBRegressor...
2026-08-04 00:22:49 | INFO | base_trainer.py | Li

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.151987 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 27946
[LightGBM] [Info] Number of data points in the train set: 49294, number of used features: 151
[LightGBM] [Info] Start training from score 93.208606


2026-08-04 00:23:00 | INFO | base_trainer.py | Line:30 | Training completed successfully.
2026-08-04 00:23:00 | INFO | base_trainer.py | Line:37 | Generating predictions using LGBMRegressor...
2026-08-04 00:23:00 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-08-04 00:23:00 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026-08-04 00:23:00 | INFO | traditional_ml_benchmark.py | Line:67 | lightgbm | RMSE=19.7637 | R2=0.7662
2026-08-04 00:23:00 | INFO | traditional_ml_benchmark.py | Line:44 | Training catboost...
2026-08-04 00:23:00 | INFO | base_trainer.py | Line:24 | Training CatBoostRegressor...
2026-08-04 00:23:58 | INFO | base_trainer.py | Line:30 | Training completed successfully.
2026-08-04 00:23:58 | INFO | base_trainer.py | Line:37 | Generating predictions using CatBoostRegressor...
2026-08-04 00:23:58 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-08-04 00:23:58 | INFO | evaluator.py | Line:80 | Evaluatio

,MAE,RMSE,R2,MAPE,Model,Training Time
0,12.898162,19.110294,0.781429,24.992359,catboost,58.100315
1,13.210585,19.763743,0.766226,26.319499,lightgbm,11.011001
2,13.359473,19.778585,0.765874,26.804319,xgboost,8.386347
3,13.449476,19.910083,0.762751,28.345515,random_forest,258.869630


In [15]:
from src.experiments.experiment_tracker import ExperimentTracker

tracker = ExperimentTracker()

tracker.save_results(results)

tracker.save_best_model(
    trainer=best_trainer,
    model_name=results.iloc[0]["Model"],
)

tracker.save_feature_importance(
    trainer=best_trainer,
    feature_names=X_train.columns,
)

tracker.save_summary(
    {
        "Best Model": results.iloc[0]["Model"],
        "MAE": float(results.iloc[0]["MAE"]),
        "RMSE": float(results.iloc[0]["RMSE"]),
        "R2": float(results.iloc[0]["R2"]),
        "MAPE": float(results.iloc[0]["MAPE"]),
        "Training Time": float(results.iloc[0]["Training Time"]),
    }
)

2026-08-04 00:30:01 | INFO | experiment_tracker.py | Line:32 | Experiment directory created at artifacts\experiments\2026-08-04_00-30-01
2026-08-04 00:30:01 | INFO | experiment_tracker.py | Line:48 | Benchmark results saved.
2026-08-04 00:30:01 | INFO | base_trainer.py | Line:59 | Model saved to artifacts\experiments\2026-08-04_00-30-01\best_model.pkl
2026-08-04 00:30:01 | INFO | experiment_tracker.py | Line:69 | Best model saved.
2026-08-04 00:30:01 | INFO | experiment_tracker.py | Line:112 | Feature importance saved.
2026-08-04 00:30:01 | INFO | experiment_tracker.py | Line:132 | Summary saved.
